# Module 5.7 — Meet AAC: a second shelter

**Module 5: Real Project — ML for Animal Adoption | Week 2 | Lecture 4**

---

You've spent the past six sessions on a single dataset: PetFinder.my. You know its quirks, its features, and how the model behaves. You found a +0.103 lift with the right features. You built a cascade. You audited it.

Now we add a second dataset: **Austin Animal Center (AAC)**, an open-data portal from a real US city shelter. About 40,000 cleaned records, dogs and cats only, no photos and no descriptions — purely tabular.

Today's question is the simplest one: **how is AAC different from PetFinder, and what does each one tell us?** Next time (5.8) we'll try to make a single model that handles both. Today we just look.

## Glossary

A handful of new terms come up in this lecture. They're shelter-domain words, not ML jargon — defining them once here saves scrolling.

- **Intake** — the event of an animal arriving at the shelter. Each row in `aac_intakes.csv` is one intake on a specific date. PetFinder doesn't have this concept; PetFinder lists are pet-by-pet, not arrival-by-arrival.
- **Outcome** — how an animal eventually left the shelter (adopted, transferred to a partner shelter, euthanized, returned to owner, etc.). Each row in `aac_outcomes.csv` matches one intake.
- **Stray** — an animal picked up off the street and brought in. No owner identified. About three-quarters of AAC intakes.
- **Owner surrender** — an animal voluntarily given up by its owner. About one-fifth of AAC intakes.
- **Distribution shift** — when the pets you want to predict on are systematically different from the pets you trained on. Different ages, different breeds, different reasons for arriving, different base rates of adoption. It's the core problem when you try to deploy a model trained on one shelter to a different shelter.

Two terms from earlier sessions show up a lot below — quick recap:

- **Rate gap** (M05-2 / M05-3) — how much higher the adoption rate is for one group vs another. Example: if 60% of young dogs are adopted in 30 days vs 35% of adult dogs, the rate gap on age is +0.25.
- **Lift** (M05-2) — how much better your model is than always predicting the majority class. Lift = accuracy − majority-class share. Lift of 0 means the model isn't helping.

In [ ]:
# Setup. Same pattern as the M05 labs that use the AAC bundle.
import os
import sys
import zipfile

AAC_URL = 'CONFIGURE_ME_AAC'     # ← your instructor will give you this Google Drive URL
AAC_DIR = '/content/aac-mini'

if not os.path.exists(f'{AAC_DIR}/aac_prep.py'):
    if AAC_URL == 'CONFIGURE_ME_AAC':
        raise RuntimeError("Set AAC_URL above to your instructor's URL.")
    os.makedirs(AAC_DIR, exist_ok=True)
    zip_path = f'{AAC_DIR}/aac-mini.zip'
    import gdown
    gdown.download(url=AAC_URL, output=zip_path, quiet=False, fuzzy=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(AAC_DIR)
    os.remove(zip_path)

sys.path.insert(0, AAC_DIR)

# Standard ML stack — same as M02 and M05-3.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# AAC-specific loader (shipped in aac-mini.zip).
from aac_prep import load_aac, AAC_SHARED, AAC_FULL, AAC_NATIVE

print('Setup OK.')


In [ ]:
df_aac = load_aac()
print(f'AAC rows: {len(df_aac):,}')
print(f'  dogs (Type==1): {(df_aac["Type"]==1).sum():,}')
print(f'  cats (Type==2): {(df_aac["Type"]==2).sum():,}')
print()
print('Some columns, first 5 rows:')
df_aac[['Type', 'Age_months', 'is_mixed_breed', 'intake_type',
        'intake_condition', 'days_in_shelter', 'outcome_type',
        'adopted_in_30_days']].head()

## AAC vs PetFinder — three big shifts

PetFinder is a website where owners and rescuers **list** their pets. AAC is a city shelter's record of every animal that walked in the door. That difference shows up in the data — the same column names, very different distributions.

Let's compare some rates side by side. The PetFinder numbers come from M05-3 (the same mini dataset you've been using all module).

In [ ]:
# PetFinder rates from M05-3 (stable across runs since the dataset is fixed).
PF_RATES = {
    'adopted_in_30': {'dogs': 0.463, 'cats': 0.549},
    'age_adult':     {'dogs': 0.296, 'cats': 0.196},
    'is_young':      {'dogs': 0.616, 'cats': 0.677},
    'gender_male':   {'dogs': 0.370, 'cats': 0.369},
    'sterilized_yes':{'dogs': 0.227, 'cats': 0.183},
    'is_mixed_breed':{'dogs': 0.865, 'cats': 0.252},
}

print(f'{"feature":>16}    {"PF dogs":>8}  {"AAC dogs":>9}    {"PF cats":>8}  {"AAC cats":>9}')
print('-' * 70)
for short, full_col in [
    ('adopted_in_30', 'adopted_in_30_days'),
    ('age_adult',     'age_adult'),
    ('is_young',      'is_young'),
    ('gender_male',   'gender_male'),
    ('sterilized_yes','sterilized_yes'),
    ('is_mixed_breed','is_mixed_breed'),
]:
    aac_d = df_aac[df_aac.Type==1][full_col].mean()
    aac_c = df_aac[df_aac.Type==2][full_col].mean()
    pf_d  = PF_RATES[short]['dogs']
    pf_c  = PF_RATES[short]['cats']
    print(f'{short:>16}     {pf_d:.3f}      {aac_d:.3f}        {pf_c:.3f}      {aac_c:.3f}')

Three things stand out:

1. **AAC dogs are much older.** 69% adult vs PetFinder's 30%, and only 23% young vs PetFinder's 62%. This is the difference between an intake record (every animal that walks in, however old) and a listing record (only the ones an owner thinks are worth photographing).
2. **AAC cats are almost all "Mix".** 98% of AAC cats have "Mix" in the breed string — Austin's data convention labels nearly every cat as "Domestic Shorthair Mix". That destroys the breed signal for cats. On PetFinder, only 25% of cats are mixed-breed.
3. **The species inversion.** On PetFinder, cats get adopted in 30 days more often than dogs (55% vs 46%). On AAC, dogs get adopted faster than cats (35% vs 21%). **The "easy" species flips.**

That third one matters most. A predictor trained on PetFinder learns that being a cat is a positive signal for fast adoption. Apply that same model to AAC and the signal is wrong — wrong direction, even. We'll see that bite us in 5.8.

## What features does AAC have that PetFinder doesn't?

PetFinder lists animal traits: age, gender, breed, vaccinated, sterilized, fur length, photo count, fee, description. Things an owner would write down on a listing form.

AAC has those (some of them), plus something PetFinder can't capture: **what was happening when the animal arrived.** Was it picked up off the street? Surrendered by its owner? Found injured? Sick?

These are **intake circumstances**, and they live in two columns:

- `intake_type` — Stray, Owner Surrender, Public Assist, Euthanasia Request
- `intake_condition` — Normal, Injured, Sick, Nursing (plus a few rare ones)

Run the next cell to see how the AAC population breaks down.

In [ ]:
print('Intake type distribution (top 4):')
print(df_aac['intake_type'].value_counts(normalize=True).round(3).head(4))
print()
print('Intake condition distribution (top 4):')
print(df_aac['intake_condition'].value_counts(normalize=True).round(3).head(4))
print()
print('Native AAC features (as binary indicators in df_aac):')
print(f'  {"feature":>26}  {"dogs rate":>10}  {"cats rate":>10}')
print('  ' + '-' * 48)
for f in ['intake_stray', 'intake_owner_surrender', 'intake_public_assist',
          'cond_normal', 'cond_injured', 'cond_sick', 'cond_nursing']:
    d = df_aac[df_aac.Type==1][f].mean()
    c = df_aac[df_aac.Type==2][f].mean()
    print(f'  {f:>26}     {d:.3f}        {c:.3f}')

## A preview of which features matter

In M05-2 / M05-3 we used **rate gaps** to find which features shift the adoption probability the most.

Same trick, new dataset. Let's look at the intake features — the ones PetFinder doesn't have.

In [ ]:
def rate_gap(df, feature, target='adopted_in_30_days'):
    # Adoption rate when feature=1 minus rate when feature=0.
    r1 = df.loc[df[feature]==1, target].mean()
    r0 = df.loc[df[feature]==0, target].mean()
    return r1 - r0

print(f'{"feature":>26}  {"dogs gap":>9}  {"cats gap":>9}')
print('-' * 50)
for f in ['intake_stray', 'intake_owner_surrender', 'cond_normal',
          'cond_injured', 'cond_sick', 'cond_nursing']:
    dg = rate_gap(df_aac[df_aac.Type==1], f)
    cg = rate_gap(df_aac[df_aac.Type==2], f)
    print(f'{f:>26}    {dg:+.3f}     {cg:+.3f}')

A few of these are worth dwelling on:

- **`intake_stray` has opposite signs by species.** Stray dogs are adopted faster (+0.10), stray cats are adopted slower (−0.22). Stray dogs are often lost pets — younger, healthier, used to humans. Stray cats are often feral or semi-feral and harder to rehome.
- **`intake_owner_surrender` is strongly positive for cats** (+0.26). Surrendered cats are typically healthy and socialized — they got along with a family until the family had to give them up. Stray cats don't carry that signal.
- **`cond_normal`** is positive for both species (+0.24 dogs, +0.15 cats). Healthy animals get adopted faster.
- **`cond_injured/sick/nursing`** are negative — the system is slow with animals that need recovery time.

Here's the takeaway, and it's the most important sentence in this lecture:

> **PetFinder's signal lives in animal traits.**
> **AAC's signal lives in intake circumstances.**

Two shelters, same prediction problem, signal in different columns. We'll come back to this — it's the architecturally important finding for 5.8.

## Does AAC itself have enough signal for a model?

Before we ask "can a PetFinder model handle AAC" (that's 5.8), the simpler question first: trained on AAC alone, predicting AAC alone, can we beat majority?

Two runs:
1. **SHARED features** — the 6 columns that map cleanly to PetFinder (age_adult, is_young, gender_male, gender_female, sterilized_yes, is_mixed_breed). What can AAC do with PetFinder's toolkit?
2. **FULL features** — SHARED plus AAC's native intake features. What can AAC do when we let it use its own signal sources?

Same model both times: standardize, fit logistic regression, 5 seeds, average.

In [ ]:
def train_simple_lr(df, features, target='adopted_in_30_days', seed=42):
    # Standardize features, fit logistic regression, return (acc, lift).
    sub = df.copy()
    X = sub[features].values.astype(float)
    y = sub[target].values.astype(int)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y,
    )
    sc = StandardScaler().fit(Xtr)
    model = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)
    yhat = model.predict(sc.transform(Xte))
    acc = accuracy_score(yte, yhat)
    majority = max(yte.mean(), 1 - yte.mean())
    return acc, acc - majority

SHARED_COLS = [f for f, _ in AAC_SHARED]
FULL_COLS   = [f for f, _ in AAC_FULL]

print(f'{"feature set":>12}  {"species":>8}  {"n feat":>6}  {"acc":>7}  {"lift":>10}')
print('-' * 55)
for cols, set_name in [(SHARED_COLS, 'SHARED'), (FULL_COLS, 'FULL')]:
    for sp, sp_name in [(1, 'dogs'), (2, 'cats')]:
        sub = df_aac[df_aac.Type==sp]
        accs, lifts = [], []
        for s in [42, 7, 2026, 1, 11]:
            acc, lift = train_simple_lr(sub, cols, seed=s)
            accs.append(acc); lifts.append(lift)
        print(f'{set_name:>12}  {sp_name:>8}  {len(cols):>6}  '
              f'{np.mean(accs):>7.3f}  {np.mean(lifts):>+10.4f}')

Two findings to read carefully:

1. **AAC SHARED: lift ~0 on both species.** With only the features PetFinder shares, AAC has essentially no extractable signal. The model is barely better than always predicting majority.

2. **AAC FULL: lift jumps to about +0.03 on dogs and +0.02 on cats.** These are small numbers, but they're real, and they come entirely from the intake features (`intake_stray`, `intake_owner_surrender`, `cond_normal`, etc.). Animal traits alone did nothing; adding intake circumstances put something on the board.

That gap between SHARED and FULL is the lesson.

PetFinder's signal lives in animal traits (age, breed, fur length, vaccinated) — M05-3 found +0.103 lift on dogs with the right features. AAC's signal lives in intake circumstances (was the animal a stray? surrendered? injured?). **Same prediction problem, same available toolkit, signal in different columns.**

That's the architecturally important finding. It's also the reason 5.8's cross-dataset model can't be a simple "train on one shelter, predict on the other." A model trained on PetFinder learns to look at animal traits. Run it on AAC, and it's looking at columns that are nearly constant on the AAC population (all adult, all mixed-breed). The signal it learned isn't there to use.

### Pushing the feature engineering on AAC

M05-3 found that on PetFinder dogs, adding `is_mixed_breed` and `is_young` pushed the lift from +0.058 to +0.103 — a ~45% relative jump from two well-chosen binary features. Does the same trick work on AAC?

Three feature-engineering moves tested on top of `AAC_FULL`, AAC same-dataset AUC (3-seed mean):

| Addition | Dogs AUC | Δ dogs | Cats AUC | Δ cats |
|---|---|---|---|---|
| baseline (AAC_FULL, 16 features) | 0.700 | — | 0.716 | — |
| + breed-content (5 dog breeds beyond pitbull / 4 cat breeds) | 0.721 | **+0.021** | 0.719 | +0.003 |
| + color content (black/white/brown/tan for dogs; plus tabby/calico/orange for cats) | 0.702 | +0.002 | 0.717 | +0.001 |
| + continuous `Age_months` (drops the `age_adult` binary) | 0.716 | **+0.017** | 0.716 | +0.000 |
| + ALL additions combined | **0.734** | **+0.035** | 0.727 | +0.011 |

Two things from this table:

1. **On AAC dogs, M05-3's feature-engineering pattern absolutely transfers.** Breed-content beyond pitbull (the original AAC investigation tested only one breed) adds +0.021 AUC. Continuous `Age_months` adds another +0.017 — replacing the 12-month binary threshold with continuous months captures finer adoption gradients than the binary cutoff did. Combined, the additions push AUC from 0.70 to 0.73 — bigger than the AAC_FULL → AAC_SHARED jump above. **The PetFinder feature-engineering pattern transfers.**

2. **On AAC cats, the same moves wash.** Every individual addition is marginal (+0.000 to +0.003). The "all additions" config barely crosses validation (+0.011). The reason: cats are 81% Domestic Shorthair (breed essentially non-informative), and the existing AAC_SHARED + NATIVE features already capture most of the predictive signal cats carry.

The same species-asymmetry shows up here as everywhere else in this lecture — **dogs are richly predictable from breed + age in a way cats aren't.** When you move M05-3's engineering moves to a new dataset, expect them to transfer on dogs and wash on cats. The translation isn't automatic.

What *does* keep washing on both species: composites of features already in scope. The AAC findings §5 tested `is_baby`, `is_senior`, `owner_surrender_normal`, `stray_normal` — all near-zero lift. The unified rule: **feature engineering that introduces information from outside the existing feature set helps; composites of in-scope features wash.**


---

## Now the harder question: can one model handle both shelters?

We've established that AAC has signal but it lives in different columns than PetFinder. Naive ideas — train on one, predict on the other; or pool both and train on the union — both fail in interesting ways. Let's look at what does and doesn't work.


### Try 1: train on PetFinder, predict on AAC

The simplest cross-dataset move: take the PetFinder model, apply it directly to AAC. What happens?

From the cross-dataset investigation (full numbers in the findings doc):

- PetFinder → AAC overall lift: **−0.133** (worse than always predicting majority)
- PetFinder → AAC dogs lift: −0.033
- PetFinder → AAC cats lift: **−0.260** (26 percentage points worse than majority)

The PetFinder model performs *worse than majority* on AAC. That's calibration mismatch in action — a model trained where 46% of cats are positive predicts too many positives on a population where only 21% are positive.


### Try 2: pool both datasets

Maybe pooling helps? Train a single model on PetFinder + AAC combined, see if it generalizes both ways.

| Source → Target | Overall lift |
|---|---|
| PetFinder → PetFinder (control) | +0.097 |
| AAC → AAC (same dataset) | 0.000 (no signal on shared features) |
| **PetFinder → AAC** | **−0.133** |
| **AAC → PetFinder** | −0.006 |
| Pooled → PetFinder | +0.048 |
| Pooled → AAC | −0.003 |

Pooling helps *neither* side. Pooled-to-PetFinder degrades from +0.097 → +0.048; pooled-to-AAC stays at majority. Domain mismatch dilutes the signal in both directions.

That's the surprising part. You'd think more data helps. It doesn't — because the data are about different populations with different signal sources.


### Try 3: multi-task with separate heads

Different architecture. Build a single network with a **shared trunk** (the part that processes the features common to both datasets) and **two dataset-specific heads** (one for PetFinder, one for AAC):

```
SHARED features (6 cols) → Trunk (64 → 32) → trunk_out (32-d)
                                                  │
                                                  ├─ + PF-specific features  → PF head  → P(adopted in 30 days | PF)
                                                  └─ + AAC-native features   → AAC head → P(adopted fast      | AAC)
```

Train both heads at once, with a combined loss. The trunk has to learn features useful for *both* head's predictions — which forces it toward representations that are invariant across shelters.


**The headline result, run on the actual code that ships in `aac-mini`:**

| Method | Dogs AUC vs §12 baseline | Cats AUC vs §12 baseline |
|---|---|---|
| PetFinder-only single-task → AAC (§12 baseline) | 0.595 | 0.574 |
| Multi-task **PF head** applied to AAC (`x_pf=0`) | +0.015 | +0.001 |
| Multi-task **trunk + linear probe** on AAC labels | **+0.066** | **+0.056** |

Two things in that table:

1. **The naive multi-task head doesn't help much.** Using the PF head with PetFinder-specific features zeroed-out only beats the §12 baseline by a hair.
2. **The trunk *does* help — meaningfully.** If you take the trunk's 32-d representation, train a fresh linear probe on AAC labels, you cross the +0.05 "real signal" threshold on *both* species.

That's the architecturally important finding. The **shared trunk learns invariances** that transfer across shelters. The dataset-specific heads memorize their dataset's features and don't generalize.


### The lesson

Multi-task isn't magic. It doesn't make a single model work on both shelters out-of-the-box. What it *does* is force the trunk into a representation that picks up cross-shelter structure — a representation that's transferable, even though the heads aren't.

Practically:

- If you want **maximum performance on PetFinder**, the single-task PetFinder model wins on PetFinder.
- If you want **maximum performance on AAC**, the AAC-only model wins on AAC.
- If you want **a representation that transfers between shelters** — for understanding what features matter generally, or for warm-starting a model on a new shelter with limited labels — the multi-task trunk is the right tool.


### The Taiwan deployment payoff

The cleanest demonstration of why this matters: applying these models to Taiwan COA dogs (a third shelter, with no training labels available).

From the Session 16 cross-dataset experiment:

| Model | Mean predicted P(30-day adopt) | Spearman vs published city rates |
|---|---|---|
| PF-only | 0.357 | 0.778 |
| AAC-only | 0.172 (collapses) | 0.556 |
| **Multi-task PF head** | **0.365** | **0.927** |

**Spearman 0.927** between the multi-task model's per-shelter Taiwan predictions and Taiwan's published city-level adoption rates. The multi-task model preserves shelter-to-shelter rankings far better than either single-shelter model — which is exactly the property you want for cross-dataset deployment.

This number is striking because the model **never saw Taiwan data during training.** The shared trunk learned a representation general enough to rank Taiwan shelters in roughly the right order, just from PetFinder + AAC.

(Why "roughly": absolute calibration is still imperfect — the model under-predicts urban shelters by ~9 percentage points — so use it for ranking and triage, not yes/no decisions. M06 covers the deployment recipe in detail.)


### Try 4 (null result): pretrain on PetFinder, fine-tune on AAC

For completeness: what if we pretrain the trunk on PetFinder, then fine-tune on AAC? Doesn't this give us the best of both?

| Config | Dogs AUC | Cats AUC |
|---|---|---|
| Pretrain-only (PF) | 0.610 | 0.572 |
| Pretrain + finetune (PF then AAC, 1K rows) | 0.614 | 0.570 |
| AAC-only on 1K rows | 0.649 | 0.620 |
| AAC-only on full ~17K rows | 0.657 | 0.621 |

Pretrain + finetune adds essentially nothing over pretrain-only. AAC-only on just 1,000 rows already beats it.

The lesson: when source and target domains have a signal-source mismatch (PF demographics vs AAC intake circumstances), domain-adaptation methods that operate on the shared feature subset cannot extract more signal than training directly on target-domain labels. The bottleneck is feature-space coverage, not training procedure.

(The investigation document calls this the "stop condition" — at this point you don't escalate to fancier domain-adaptation methods. The signal-source asymmetry is too fundamental.)


### A deeper question: what should we actually predict?

We've trained against `adopted_in_30_days` (PetFinder's AdoptionSpeed ≤ 2) all module. That 30-day cutoff was chosen because it splits PetFinder roughly 50/50 and is easy to talk about. AAC suggests we could be predicting better — but the right target choice depends on species and operational goal.

**Three targets the AAC data exposes, with empirical evidence for each:**

**1. `ever_adopted`** — binary, longest time window. AAC: `outcome_type == 'Adoption'`. PetFinder: `AdoptionSpeed != 4`. The same semantic question, answerable on both datasets.

- **Cats benefit substantially.** PF→AAC cross-shelter AUC: 0.569 (`adopted_in_30`) → **0.634** (`ever_adopted`). The longer window integrates over per-shelter pacing differences that the 30-day cutoff exposes.
- **Dogs don't.** Same comparison on dogs: 0.584 → 0.550 — slightly worse. The 30-day cutoff happens to land near dogs' actual fast/slow boundary; lengthening dilutes that signal. Mirrors the cross-species inversion seen elsewhere in the investigation.
- **Use it:** for cats cross-shelter deployment. Stick with `adopted_in_30` for dogs cross-shelter.

**2. Per-shelter median cutoff (`adopted_fast_AAC`)** — each shelter's own median time-to-adoption among adopted animals defines the FAST threshold. PetFinder: ~30 days both species. AAC: 7 days dogs, 26 days cats.

- **Dogs benefit modestly.** AAC same-dataset AUC: 0.700 (`adopted_in_30`) → **0.722** (`adopted_fast_AAC`). The 7-day cutoff is much tighter than 30 days and captures finer fast-adopter structure.
- **Cats barely change.** 0.716 → 0.722 — cats' median is ~26 days, target is essentially the same as `adopted_in_30`.
- **Use it:** for within-AAC dogs deployment. For cats either target is equivalent.

**3. Outcome categories (5-class softmax)** — predict the actual outcome class: `{Adoption, Transfer, Return-to-Owner, Euthanasia, Died}`. AAC has these labels directly; PetFinder doesn't (it only knows about speed of adoption, not the reasons for non-adoption).

- **Both species benefit substantially.** Dogs accuracy **0.554** vs majority 0.424 (lift **+0.130**); cats accuracy **0.598** vs majority 0.500 (lift **+0.098**).
- **Multiple classes recoverable per species.** Dogs: Adoption recall 0.82, RTO 0.62. Cats: Adoption 0.70, Transfer 0.58, Euthanasia 0.53. Died is too rare (0.8%) to recover on either species.
- **Use it:** for AAC-specific deployment where the *reason* for non-adoption matters — transfer planning, euthanasia avoidance, return-to-owner tracking.

**The deeper pattern across all three:** target choice is a hyperparameter that interacts with species structure and operational goal. `adopted_in_30` isn't wrong — it's a starting point. When the dataset offers richer targets (longer time windows, per-shelter calibrations, multi-class outcomes), **test them.** The right target may differ across species or across operational decisions.

What *doesn't* work, based on the M05 investigation: reaching for richer loss functions (continuous time-to-adoption regression, Cox proportional hazards, multi-cutoff multi-task). All four tied binary BCE at the right cutoff. **The feature set caps the signal; the target framing matters; the loss function does not.**


### What we built

By the end of this lecture and 5.8's hands-on lab you'll have:

- A working **single-task model** for PetFinder (you built this in 5.2 / 5.4)
- A working **single-task model** for AAC (the same MLP, retrained on AAC's intake features — Strategy G)
- A **multi-task model** with a shared trunk and two heads
- A **trunk-only linear probe** that extracts the shared representation and adapts it to a new domain
- A path to **Taiwan deployment** via the multi-task PF head — the recipe M06 runs end-to-end

That's a real cross-dataset toolkit — most ML projects never explicitly build this; they retrain from scratch each time. The multi-task pattern is a nice tool to have in your back pocket.


### What's next (5.8, lab)

The 5.8 lab is the hands-on. You'll:

1. Reproduce the PetFinder→AAC direct-transfer failure
2. Train the multi-task model (shared trunk + two heads) on PetFinder + AAC
3. Extract the trunk, train a linear probe on AAC labels — verify the +0.066 / +0.056 finding for yourself
4. Interpret *why* the trunk transfers but the heads don't — write it in your own words
5. (Stretch) Run the pretrain-finetune null result — see for yourself why it doesn't help

Taiwan COA deployment — scoring Taiwan dogs through the multi-task PF head, comparing to published city rates — lands in **M06** with the full toolkit (multi-task + photos + deployment recipe). M05's job was to build the architecture; M06 ships it.

Same toolkit as today's lecture, but you write the multi-task training loop and the linear probe yourself.

After 5.8 the M05 module closes (5.9–5.12 is scandal, audit, sourcing, and the toolkit recap that hands off to M06). M06 takes the multi-task model and adds the photos, the Taiwan deployment, and your dog's final brief.
